In [ ]:
# ── Cell 1: Load config and read sales_summary from S3 ───────────────────────
exec(open("/Workspace/ecommerce_retail/config.py").read())

df_sales = spark.read \
    .option("fs.s3a.access.key", ACCESS_KEY) \
    .option("fs.s3a.secret.key", SECRET_KEY) \
    .option("fs.s3a.session.token", SESSION_TOKEN) \
    .option("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider") \
    .parquet(f"{S3_GOLD_BASE}/sales_summary/")

df_sales.createOrReplaceTempView("sales_summary")
print(f"✅ {df_sales.count()} rows loaded")
df_sales.printSchema()

In [ ]:
# ── Cell 2: Install libraries ─────────────────────────────────────────────────
%pip install plotly pandas

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# ── Cell 4: KPI Summary Table ─────────────────────────────────────────────────
df = spark.sql("""
    SELECT
        SUM(total_orders)                    AS total_orders,
        SUM(unique_customers)                AS total_unique_customers,
        ROUND(SUM(gross_revenue), 2)         AS total_gross_revenue,
        ROUND(SUM(net_revenue), 2)           AS total_net_revenue,
        ROUND(SUM(total_discounts), 2)       AS total_discounts,
        ROUND(SUM(total_tax), 2)             AS total_tax_collected,
        ROUND(SUM(shipping_cost), 2)         AS total_shipping_cost,
        SUM(total_units_sold)                AS total_units_sold,
        ROUND(AVG(avg_order_value), 2)       AS avg_order_value,
        ROUND(AVG(avg_fulfillment_days), 2)  AS avg_fulfillment_days
    FROM sales_summary
""").toPandas()

kpi = df.T.reset_index()
kpi.columns = ["KPI", "Value"]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>KPI</b>", "<b>Value</b>"],
        fill_color="#2E3A59",
        font=dict(color="white", size=13),
        align="left",
        height=35
    ),
    cells=dict(
        values=[kpi["KPI"], kpi["Value"]],
        fill_color=[["#F1EFE8", "#FFFFFF"] * len(kpi)],
        font=dict(size=12),
        align="left",
        height=30
    )
)])
fig.update_layout(title="📊 Sales Summary — KPI Overview")
fig.show()

In [ ]:
# ── Cell 5: Chart 1 — Monthly Gross vs Net Revenue (Line) ────────────────────
df = spark.sql("""
    SELECT
        order_year,
        order_month,
        CONCAT(CAST(order_year AS STRING), '-',
               LPAD(CAST(order_month AS STRING), 2, '0')) AS year_month,
        ROUND(SUM(gross_revenue), 2)  AS gross_revenue,
        ROUND(SUM(net_revenue), 2)    AS net_revenue,
        ROUND(SUM(total_discounts), 2) AS total_discounts
    FROM sales_summary
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
""").toPandas()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["year_month"], y=df["gross_revenue"],
    mode="lines+markers", name="Gross Revenue",
    line=dict(color="#636EFA", width=2)
))
fig.add_trace(go.Scatter(
    x=df["year_month"], y=df["net_revenue"],
    mode="lines+markers", name="Net Revenue",
    line=dict(color="#2ECC71", width=2)
))
fig.add_trace(go.Scatter(
    x=df["year_month"], y=df["total_discounts"],
    mode="lines+markers", name="Total Discounts",
    line=dict(color="#E74C3C", width=2, dash="dash")
))
fig.update_layout(
    title="📈 Monthly Gross Revenue vs Net Revenue vs Discounts",
    xaxis_title="Month",
    yaxis_title="Amount ($)",
    legend_title="Metric",
    xaxis_tickangle=-45
)
fig.show()

In [ ]:
# ── Cell 6: Chart 2 — Monthly Orders and Unique Customers (Dual Line) ─────────
df = spark.sql("""
    SELECT
        order_year,
        order_month,
        CONCAT(CAST(order_year AS STRING), '-',
               LPAD(CAST(order_month AS STRING), 2, '0')) AS year_month,
        SUM(total_orders)      AS total_orders,
        SUM(unique_customers)  AS unique_customers
    FROM sales_summary
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
""").toPandas()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Monthly Total Orders", "Monthly Unique Customers")
)
fig.add_trace(
    go.Scatter(
        x=df["year_month"], y=df["total_orders"],
        mode="lines+markers", name="Orders",
        line=dict(color="#636EFA", width=2),
        fill="tozeroy"
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=df["year_month"], y=df["unique_customers"],
        mode="lines+markers", name="Customers",
        line=dict(color="#EF553B", width=2),
        fill="tozeroy"
    ),
    row=1, col=2
)
fig.update_layout(
    title_text="🛒 Monthly Orders vs Unique Customers",
    showlegend=False,
    height=400
)
fig.update_xaxes(tickangle=-45)
fig.show()

In [ ]:
# ── Cell 7: Chart 3 — Revenue by Year (Bar) ───────────────────────────────────
df = spark.sql("""
    SELECT
        order_year,
        ROUND(SUM(gross_revenue), 2)   AS gross_revenue,
        ROUND(SUM(net_revenue), 2)     AS net_revenue,
        SUM(total_orders)              AS total_orders,
        SUM(total_units_sold)          AS total_units_sold
    FROM sales_summary
    GROUP BY order_year
    ORDER BY order_year
""").toPandas()

df["order_year"] = df["order_year"].astype(str)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df["order_year"], y=df["gross_revenue"],
    name="Gross Revenue",
    marker_color="#636EFA",
    text=df["gross_revenue"],
    textposition="outside"
))
fig.add_trace(go.Bar(
    x=df["order_year"], y=df["net_revenue"],
    name="Net Revenue",
    marker_color="#2ECC71",
    text=df["net_revenue"],
    textposition="outside"
))
fig.update_layout(
    barmode="group",
    title="📅 Gross vs Net Revenue by Year",
    xaxis_title="Year",
    yaxis_title="Revenue ($)",
    legend_title="Metric"
)
fig.show()

In [ ]:
# ── Cell 8: Chart 4 — Monthly Units Sold (Bar) ────────────────────────────────
df = spark.sql("""
    SELECT
        CONCAT(CAST(order_year AS STRING), '-',
               LPAD(CAST(order_month AS STRING), 2, '0')) AS year_month,
        SUM(total_units_sold) AS total_units_sold
    FROM sales_summary
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
""").toPandas()

fig = px.bar(
    df,
    x="year_month",
    y="total_units_sold",
    title="📦 Monthly Units Sold",
    color="total_units_sold",
    color_continuous_scale="Blues",
    text="total_units_sold"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Units Sold",
    xaxis_tickangle=-45
)
fig.show()

In [ ]:
# ── Cell 9: Chart 5 — Avg Order Value Trend (Line) ───────────────────────────
df = spark.sql("""
    SELECT
        CONCAT(CAST(order_year AS STRING), '-',
               LPAD(CAST(order_month AS STRING), 2, '0')) AS year_month,
        ROUND(AVG(avg_order_value), 2)      AS avg_order_value,
        ROUND(AVG(avg_fulfillment_days), 2) AS avg_fulfillment_days
    FROM sales_summary
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
""").toPandas()

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Avg Order Value ($)", "Avg Fulfillment Days"),
    shared_xaxes=True
)
fig.add_trace(
    go.Scatter(
        x=df["year_month"], y=df["avg_order_value"],
        mode="lines+markers", name="Avg Order Value",
        line=dict(color="#636EFA", width=2)
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=df["year_month"], y=df["avg_fulfillment_days"],
        mode="lines+markers", name="Avg Fulfillment Days",
        line=dict(color="#EF553B", width=2)
    ),
    row=2, col=1
)
fig.update_layout(
    title_text="⏱️ Avg Order Value & Fulfillment Days Over Time",
    showlegend=False,
    height=500
)
fig.show()

In [ ]:
# ── Cell 10: Chart 6 — Revenue Breakdown Donut (Gross components) ─────────────
df = spark.sql("""
    SELECT
        ROUND(SUM(net_revenue), 2)     AS net_revenue,
        ROUND(SUM(total_discounts), 2) AS total_discounts,
        ROUND(SUM(total_tax), 2)       AS total_tax,
        ROUND(SUM(shipping_cost), 2)   AS shipping_cost
    FROM sales_summary
""").toPandas()

labels = ["Net Revenue", "Discounts", "Tax", "Shipping"]
values = [
    df["net_revenue"].values[0],
    df["total_discounts"].values[0],
    df["total_tax"].values[0],
    df["shipping_cost"].values[0]
]

fig = go.Figure(data=[go.Pie(
    labels=labels,
    values=values,
    hole=0.4,
    marker=dict(colors=["#2ECC71", "#E74C3C", "#F39C12", "#3498DB"]),
    textinfo="percent+label+value"
)])
fig.update_layout(title="💰 Gross Revenue Breakdown — Net vs Discounts vs Tax vs Shipping")
fig.show()

In [ ]:
# ── Cell 11: Chart 7 — Monthly Revenue Heatmap (Year × Month) ─────────────────
df = spark.sql("""
    SELECT
        order_year,
        order_month,
        ROUND(SUM(gross_revenue), 2) AS gross_revenue
    FROM sales_summary
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
""").toPandas()

df["order_year"]  = df["order_year"].astype(str)
df["order_month"] = df["order_month"].astype(str)

pivot = df.pivot(index="order_year", columns="order_month", values="gross_revenue").fillna(0)

fig = px.imshow(
    pivot,
    title="🔥 Monthly Revenue Heatmap (Year × Month)",
    color_continuous_scale="Blues",
    text_auto=True,
    aspect="auto",
    labels=dict(x="Month", y="Year", color="Revenue ($)")
)
fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Year"
)
fig.show()

In [ ]:
# ── Cell 12: Chart 8 — Shipping Cost vs Tax vs Discount Monthly (Stacked Area) ─
df = spark.sql("""
    SELECT
        CONCAT(CAST(order_year AS STRING), '-',
               LPAD(CAST(order_month AS STRING), 2, '0')) AS year_month,
        ROUND(SUM(total_discounts), 2) AS total_discounts,
        ROUND(SUM(total_tax), 2)       AS total_tax,
        ROUND(SUM(shipping_cost), 2)   AS shipping_cost
    FROM sales_summary
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
""").toPandas()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["year_month"], y=df["total_discounts"],
    mode="lines", name="Discounts",
    stackgroup="one", fill="tonexty",
    line=dict(color="#E74C3C")
))
fig.add_trace(go.Scatter(
    x=df["year_month"], y=df["total_tax"],
    mode="lines", name="Tax",
    stackgroup="one", fill="tonexty",
    line=dict(color="#F39C12")
))
fig.add_trace(go.Scatter(
    x=df["year_month"], y=df["shipping_cost"],
    mode="lines", name="Shipping",
    stackgroup="one", fill="tonexty",
    line=dict(color="#3498DB")
))
fig.update_layout(
    title="📉 Monthly Discounts vs Tax vs Shipping Cost (Stacked)",
    xaxis_title="Month",
    yaxis_title="Amount ($)",
    legend_title="Cost Type",
    xaxis_tickangle=-45
)
fig.show()